In [5]:
import subprocess
import sys
import time
from pathlib import Path

# ---------- Configuration ----------

APPLICATIONS = {
    # "cpu_bound": {
    #     "locustfile": "cpu_bound/locustfile.py",
    #     "port": 8081,
    # },
    "memory_bound": {
        "locustfile": "memory_bound/locustfile.py",
        "port": 8083,
    },
    # "io_bound": {
    #     "locustfile": "io_bound/locustfile.py",
    #     "port": 8083,
    # },
}

C = "CM"

LOADS = [1000, 1500, 2000, 2500]
MAX_LOAD = 4000
TOTAL_TIME = 120  # seconds
RAMP_UP = 40       # users spawned per second
COOLDOWN = 60      # seconds between tests

METRICS_DURATION = TOTAL_TIME + 5

RESULTS_DIR = Path(f"results/with_ramp_up/{C}")


def run_test(scenario, application, config):
    test_name = f"{scenario}_{application}"

    locustfile = config["locustfile"]
    host = f"http://127.0.0.1:{config['port']}"

    csv_prefix = RESULTS_DIR / test_name
    metrics_path = RESULTS_DIR / f"{test_name}_vmstat.txt"
    log_path = RESULTS_DIR / f"{test_name}.log"

    # Monitor the LOCAL machine, without SSH.
    metrics_command = [
        "vmstat", "-t", "-n", "1"
    ]

    # timeout is applied to the monitoring process only.
    metrics_command = [
        "timeout", f"{METRICS_DURATION}s",
        *metrics_command,
    ]

    locust_command = [
        sys.executable, "-m", "locust",
        "-f", locustfile,
        "--headless",
        "--host", host,
        "--users", str(MAX_LOAD),
        "--spawn-rate", str(RAMP_UP),
        "--run-time", f"{TOTAL_TIME}s",
        "--csv", str(csv_prefix),
        "--csv-full-history",
    ]

    print(f"\n{'=' * 60}")
    print(f"Scenario:     {scenario}")
    print(f"Application:  {application}")
    print(f"Locust file:  {locustfile}")
    print(f"Target:       {host}")
    print(f"Users:        {MAX_LOAD}")
    print(f"Spawn rate:   {RAMP_UP} users/s")
    print(f"Duration:     {TOTAL_TIME}s")
    print(f"{'=' * 60}")

    with open(metrics_path, "w") as metrics_file:
        metrics_process = subprocess.Popen(
            metrics_command,
            stdout=metrics_file,
            stderr=subprocess.STDOUT,
            text=True,
        )

        # Allow vmstat to start collecting before the load begins.
        time.sleep(1)

        try:
            with open(log_path, "w") as log:
                result = subprocess.run(
                    locust_command,
                    stdout=log,
                    stderr=subprocess.STDOUT,
                    text=True,
                )
        finally:
            # Stop monitoring if it is still running.
            metrics_process.terminate()

            try:
                metrics_process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                metrics_process.kill()
                metrics_process.wait()

    if result.returncode != 0:
        print(f"ERROR: {test_name} failed. Check {log_path}")
        return False

    print(f"Completed: {test_name}")
    print(f"Locust log: {log_path}")
    print(f"VM metrics: {metrics_path}")

    return True


def main():
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    scenario = C
    failures = []
    completed = 0

    applications = list(APPLICATIONS.items())

    print(f"\nScenario: {scenario}")
    print(f"Planned tests: {len(applications)}")
    print(f"Results directory: {RESULTS_DIR.resolve()}")

    for index, (application, config) in enumerate(applications):
        # Cooldown between tests, but not before the first one.
        if index > 0:
            print(f"\nCooling down for {COOLDOWN}s...")
            time.sleep(COOLDOWN)

        success = run_test(scenario, application, config)

        if success:
            completed += 1
        else:
            failures.append(f"{scenario}_{application}")

    print("\nAll tests processed.")
    print(f"Successful: {completed}/{len(applications)}")

    if failures:
        print("Failed tests:")
        for failure in failures:
            print(f"  - {failure}")
    else:
        print("All tests completed successfully.")

    print(f"Results saved in: {RESULTS_DIR.resolve()}")


if __name__ == "__main__":
    main()


Scenario: CM
Planned tests: 1
Results directory: /home/mignoe/Documents/git/Roteiro2-VirtualBox/results/with_ramp_up/CM

Scenario:     CM
Application:  memory_bound
Locust file:  memory_bound/locustfile.py
Target:       http://127.0.0.1:8083
Users:        4000
Spawn rate:   40 users/s
Duration:     120s
Completed: CM_memory_bound
Locust log: results/with_ramp_up/CM/CM_memory_bound.log
VM metrics: results/with_ramp_up/CM/CM_memory_bound_vmstat.txt

All tests processed.
Successful: 1/1
All tests completed successfully.
Results saved in: /home/mignoe/Documents/git/Roteiro2-VirtualBox/results/with_ramp_up/CM
